In [1]:
!pip install pypdf

     |████████████████████████████████| 297 kB 1.3 MB/s eta 0:00:01


In [2]:
!pip install langchain_community

In [3]:
!pip install pinecone-client

     |████████████████████████████████| 244 kB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 85 kB 5.7 MB/s  eta 0:00:01


In [4]:
!pip install -U langchain_ollama

Requirement already up-to-date: langchain_ollama in /home/yash/.local/lib/python3.8/site-packages (0.1.3)


In [5]:
!pip install pyngrok

In [6]:
!pip install tiktoken

     |████████████████████████████████| 1.1 MB 881 kB/s eta 0:00:01


In [7]:
!pip install langchain

# DATA Preprocessing

In [ ]:
# Reading the data from directory.
from langchain_community.document_loaders import PyPDFLoader
from os import path
from glob import glob

pages = []
files = glob(path.join("/content/Data", "*.pdf"))
for file in files:
    loader = PyPDFLoader(file)
    pages.extend(loader.load_and_split())

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
docs = text_splitter.split_documents(pages)

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = [embedding_model.encode(doc.page_content) for doc in docs]

/home/yash/.local/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2024-12-30 20:02:39.861001: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-30 20:02:41.342624: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/yash/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`

Initilazing the Vector Database (pinecone)

In [12]:
import pinecone
# pinecone_api_key = "<-- Use API key from Pinecone -->"
index_name = "rag-qa-db"

# Initialize Pinecone client
pc = pinecone.Pinecone(api_key=pinecone_api_key)
# 
# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(vectors[0]),  # size of Embedding of Sentence Transformer model
        metric="cosine",
        spec=pinecone.ServerlessSpec(
            cloud="aws",
            region="us-east-1"  # region Specifing
        )
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

index = pc.Index(index_name)

Index 'rag-qa-db' already exists.


Update the Database

In [ ]:
vectors = [vector.tolist() for vector in vectors]

In [ ]:
indexed_vectors = [
    (f"vec_{i}", vector, {"text": texts[i]})  # Replace "texts[i]" with actual metadata if needed
    for i, vector in enumerate(vectors)
]


In [ ]:
index.upsert(indexed_vectors)

{'upserted_count': 762}

In [9]:
def retrieve_relevant_docs(query, top_k=3):
    query_embedding = embedding_model.encode(query).tolist()
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)
    return [match["metadata"]["text"] for match in results["matches"]]

In [13]:
query = 'Privacy Policy'

In [6]:
query_embedding = embedding_model.encode(query).tolist()
index.query(vector=query_embedding, top_k=4, include_metadata=True)


{'matches': [{'id': 'vec_583',
              'metadata': {'text': 'Privacy Policy here if we decide to do so. '
                                   'In view of the p ossibility that we may '
                                   'change this \n'
                                   'Privacy Policy at any time without '
                                   'informing you, we suggest that you review '
                                   'it periodically. \n'
                                   'Whenever any information is collected, our '
                                   'rights to use it will be determined by the '
                                   'privacy \n'
                                   'policy that is in effect at the time. \n'
                                   '5) Contact \n'
                                   ' \n'
                                   '               Please send any questions, '
                                   'comments or requests regarding this '
              

In [53]:
retriver = retrieve_relevant_docs(query)
retriver

['Privacy Policy here if we decide to do so. In view of the p ossibility that we may change this \nPrivacy Policy at any time without informing you, we suggest that you review it periodically. \nWhenever any information is collected, our rights to use it will be determined by the privacy \npolicy that is in effect at the time. \n5) Contact \n \n               Please send any questions, comments or requests regarding this privacy policy to our  \nprivacy  team at info@wsfx.in',
 'business which we may enter into with you) disclose your personal data to: \n\uf0b7 any person or entity to whom we are required or requested to make such disclosure \nby any court of competent jurisdiction or by any governmental, taxation or other \nregulatory authority, law enforcement agency or similar body; \n\uf0b7 our professional advisers or consultants, including lawyers, bankers, auditors, \naccountants and insurers providing consultancy, legal, banking, audit, accounting or \ninsurance services to us;

In [14]:
from langchain_community.llms import Ollama
llm_model = Ollama(model="llama2")

In [15]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context.
Think step by step before providing the detailed answer
<context>
{context}
</context>
Question :{input} """)

In [16]:
from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm_model, prompt)

In [17]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
def embedding_function(text):
    return embedding_model.encode(text).tolist()

/home/yash/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [18]:
from langchain.vectorstores import Pinecone
vectorstore = Pinecone(index=index , embedding=embedding_function ,text_key="text")
retriever = vectorstore.as_retriever()

/home/yash/.local/lib/python3.8/site-packages/langchain_community/vectorstores/pinecone.py:68: UserWarning: Passing in `embedding` as a Callable is deprecated. Please pass in an Embeddings object instead.
  warnings.warn(


In [19]:
from langchain.chains import create_retrieval_chain
qa_chain = create_retrieval_chain( retriever , document_chain)

In [22]:
Response = qa_chain.invoke({"input":"Enlist the Customer-support service."})

In [23]:
print(Response['answer'])

Based on the provided context, the customer-support service can be enlisted as follows:

1. Contact: Refers to the initial interaction between the customer and the support service.
2. Contact Person: The individual who is the primary point of contact for the customer.
3. Occurred Problems: Issues that arise during the contact, such as technical problems or misunderstandings.
4. Assessment: Evaluation of the problem and its impact on the relationship between the customer and the support service.
5. Outcome: The result of the assessment, which may include resolving the issue or escalating it to other departments.
6. Effect on Relationship: The impact of the outcome on the customer's perception of the support service and the overall relationship between the customer and the telecommunications provider.
7. Technical Issues: Problems related to the technical aspects of the core service, such as network coverage, speed, or connectivity.
8. Customer Concerns: Issues that are not related to th

In [26]:
Response = qa_chain.invoke({"input":"What payment methods do you accept, and how can I update my billing information?"})

In [27]:
print(Response['answer'])

 Based on the provided context, the answer to the question "What payment methods do you accept, and how can I update my billing information?" is:

The University accepts payment through Rowan University credit and debit cards. To update your billing information, please contact the Office of Contracting & Procurement for assistance.

Explanation:

Based on the context provided, Rowan University has set up workflow approvals for Amazon Business orders, and the preferred payment method is Rowan University credit and debit cards. Therefore, if you need to update your billing information, you should contact the Office of Contracting & Procurement for assistance.

Additionally, the context mentions that Pay by Invoice is not a valid payment method, so it's important to select the correct payment method when placing an order through Amazon Business.
